### Application 3: Offensive speech detector
Identifies the presence of offensive speech within the feedback dataset.

This program is executed on google colab using a T4 GPU. To run, upload the `offensive_speech_detector` model and this notebook, `offensive_speech_detector`, to google colab.

In [ ]:
!pip uninstall -y unsloth unsloth_zoo
!pip -q install -U pip setuptools wheel
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 70.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 169.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 64.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 92.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 153.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 140.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 149.5 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
 

In [ ]:
feedback_list = [
    "Assignments were helpful for understanding the concepts better.",
    "The course was well-structured and organized.",
    "These lectures are so boring and a complete waste of time.",
    "The assignments are stupid and make no sense.",
    "The teaching is absolutely terrible and confusing.",
    "This course is a joke, I didn't learn anything useful.",
    "The instructions are so bad, it's really frustrating."
]

In [ ]:
# connect to google colab to retrieve fine-tuned llama 3 model
from google.colab import drive
drive.mount('/content/drive')
save_dir = "/content/drive/My Drive/offensive_speech_detector"

In [ ]:
from unsloth import FastLanguageModel
import torch
from peft import PeftModel
import ast
import json

max_seq_length = 2048

# load fine-tuned llama 3 model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = torch.float16,
    load_in_4bit = True,
    device_map = "cuda",
)
model = PeftModel.from_pretrained(
    model,
    save_dir,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Mounted at /content/drive
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
from unsloth import FastLanguageModel
from tqdm import tqdm
import torch
import json
import re
import ast

FastLanguageModel.for_inference(model)
model.eval()

# define alpaca prompt template for inference
alpaca_prompt = """### Instruction:
{}

### Input:
{}

### Response:
{}"""


for i in range(len(feedback_list)):
    instruction = f"""You are a classifier for offensive speech in student feedback. Return ONLY 0 or 1. Use 1 if comment includes offensive speech. Use 0 if comment does not contain offensive speech."""
    sentence = feedback_list[i]
    prompt = alpaca_prompt.format(instruction,sentence,"",)
    inputs = tokenizer([prompt],return_tensors="pt",).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=False,
            do_sample=False,
        )

    # process output
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    label = decoded.split("### Response:")[-1].strip()[0]
    if label == "0":
      print(f"The comment '{sentence}' does not contain offensive speech.")
    elif label == "1":
      print(f"The comment '{sentence}' contains offensive speech.")
    else:
      print("It could not be determined whether the sentence contains offensive speech.")


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The comment 'Assignments were helpful for understanding the concepts better.' does not contain offensive speech.


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The comment 'The course was well-structured and organized.' does not contain offensive speech.


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

The comment 'The assignments are stupid and make no sense.' contains offensive speech.


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The comment 'The teaching is absolutely terrible and confusing.' contains offensive speech.


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The comment 'This course is a joke, I didn't learn anything useful.' contains offensive speech.
The comment 'The instructions are so bad, it's really frustrating.' contains offensive speech.
